In [1]:
%pip install pymongo

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: pymongo in c:\users\enzod\onedrive\área de trabalho\neroai\hubot\env\lib\site-packages (4.10.1)




[notice] A new release of pip available: 22.3.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pymongo import MongoClient
from datetime import datetime
import os
from dotenv import load_dotenv

MONGODB_ATLAS_CLUSTER_URI = os.getenv('MONGODB_URI')
# Conectar ao MongoDB
client = MongoClient(MONGODB_ATLAS_CLUSTER_URI)
db = client["hubot"]
collection = db["hub-mesas"]

In [ ]:
# Função para atualizar o status de uma mesa
def atualizar_status_mesa(mesa_id, novo_status, user_id=None):
    update = {
        "status": novo_status,
        "ultimo_update": datetime.utcnow()
    }
    if user_id:
        update["reservado_por"] = user_id
    else:
        update["reservado_por"] = None
    collection.update_one({"_id": mesa_id}, {"$set": update})

In [14]:
# Função para contar mesas disponíveis
def contar_mesas_disponiveis():
    return collection.count_documents({"status": "disponível"})


In [13]:
# Função para listar os números das mesas disponíveis
def listar_mesas_disponiveis():
    mesas_disponiveis = collection.find({"status": "disponível"}, {"mesa": 1, "_id": 0})
    return [mesa["mesa"] for mesa in mesas_disponiveis]


In [22]:
# Função para reservar uma mesa específica
def reservar_mesa(mesa_id, user_id):
    resultado = collection.update_one(
        {"mesa": mesa_id, "status": "disponível"},  # Garantir que a mesa está disponível
        {"$set": {"status": "ocupada", "reservado_por": user_id, "ultimo_update": datetime.utcnow()}}
    )
    if resultado.modified_count > 0:
        print(f"A mesa {mesa_id} foi reservada com sucesso para o usuário {user_id}.")
        return True
    else:
        print(f"A mesa {mesa_id} não está disponível para reserva.")
        return False


In [23]:
# Exemplo de fluxo completo no chatbot
def fluxo_reserva_mesa(user_id):
    # Passo 1: Informar quantas mesas estão disponíveis
    num_disponiveis = contar_mesas_disponiveis()
    print(f"Atualmente, há {num_disponiveis} mesas disponíveis.")

    # Passo 2: Listar mesas disponíveis para o usuário escolher
    mesas = listar_mesas_disponiveis()
    print("Mesas disponíveis:", mesas)
    
    if mesas:
        try:
            # Solicitar que o usuário escolha uma mesa entre as disponíveis
            mesa_escolhida = int(input("Qual mesa você deseja escolher? (Digite o número da mesa): "))
            # Verificar se a mesa escolhida está realmente disponível
            if mesa_escolhida in mesas:
                # Passo 3: Reservar a mesa escolhida
                sucesso = reservar_mesa(mesa_escolhida, user_id)
                if sucesso:
                    print(f"Mesa {mesa_escolhida} reservada com sucesso.")
                else:
                    print(f"Desculpe, a mesa {mesa_escolhida} não pôde ser reservada.")
            else:
                print("A mesa escolhida não está disponível. Por favor, escolha uma mesa válida.")
        except ValueError:
            print("Entrada inválida. Por favor, insira o número da mesa.")
    else:
        print("Não há mesas disponíveis para reserva no momento.")


In [29]:
# Função para liberar uma mesa específica
def liberar_mesa(mesa_id):
    resultado = collection.update_one(
        {"mesa": mesa_id, "status": "ocupada"},  # Garante que a mesa está ocupada antes de liberar
        {"$set": {"status": "disponível", "reservado_por": "", "data_update": datetime.utcnow()}}
    )
    
    if resultado.modified_count > 0:
        print(f"Mesa {mesa_id} liberada com sucesso.")
        return True
    else:
        print(f"A mesa {mesa_id} não está ocupada ou não foi encontrada.")
        return False


In [30]:
liberar_mesa(2)

Mesa 2 liberada com sucesso.


True